# Netflix Movie Recommendation System

This notebook builds a collaborative-filtering recommendation system using
Netflix Prize-style ratings data and Singular Value Decomposition (SVD).

The workflow has been cleaned into a portfolio-ready structure:

**raw data → parsing → EDA → activity filtering → SVD evaluation → final model → Top-N recommendations**


## 1. Imports


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


## 2. File Paths

Place the raw files in the repository's `data/` directory.

This removes the Google Colab / Google Drive dependency from the original notebook.


In [ ]:
DATA_DIR = Path("../data")

RATINGS_PATH = DATA_DIR / "combined_data_1.txt"
TITLES_PATH = DATA_DIR / "movie_titles.csv"

if not RATINGS_PATH.exists():
    raise FileNotFoundError(
        "Missing data/combined_data_1.txt. See data/README.md."
    )

if not TITLES_PATH.exists():
    raise FileNotFoundError(
        "Missing data/movie_titles.csv. See data/README.md."
    )


## 3. Parse Netflix Ratings Data

Netflix Prize-style ratings files contain a movie-ID header such as `1:` followed by
customer-rating rows. The function below converts that layout into a tidy table.


In [ ]:
def parse_netflix_ratings(path):
    rows = []
    current_movie_id = None

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            if line.endswith(":"):
                current_movie_id = int(line[:-1])
                continue

            parts = line.split(",")
            if len(parts) < 2 or current_movie_id is None:
                continue

            customer_id = int(parts[0])
            rating = float(parts[1])

            rows.append((customer_id, current_movie_id, rating))

    return pd.DataFrame(
        rows,
        columns=["Cust_Id", "Movie_Id", "Rating"]
    )


ratings = parse_netflix_ratings(RATINGS_PATH)
ratings.head()


## 4. Dataset Overview


In [ ]:
print(f"Rows: {len(ratings):,}")
print(f"Unique customers: {ratings['Cust_Id'].nunique():,}")
print(f"Unique movies: {ratings['Movie_Id'].nunique():,}")

ratings.info()


In [ ]:
ratings["Rating"].describe()


## 5. Ratings Distribution


In [ ]:
rating_counts = (
    ratings["Rating"]
    .value_counts()
    .sort_index()
)

rating_counts


In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(
    x=rating_counts.index.astype(int),
    y=rating_counts.values
)
plt.title("Distribution of Netflix Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Ratings")
plt.show()


## 6. User and Movie Activity

Recommendation datasets are usually sparse. Very inactive users and very rarely rated
movies can make experimentation slower and less stable, so we profile activity before filtering.


In [ ]:
movie_activity = (
    ratings.groupby("Movie_Id")["Rating"]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
)

customer_activity = (
    ratings.groupby("Cust_Id")["Rating"]
    .agg(["count", "mean"])
    .sort_values("count", ascending=False)
)

movie_activity.head()


In [ ]:
customer_activity.head()


## 7. Activity Filtering

The original notebook used the 70th percentile of rating counts as a benchmark.
This cleaned version keeps that idea but makes the thresholds explicit and reproducible.


In [ ]:
ACTIVITY_QUANTILE = 0.70

movie_threshold = int(
    round(movie_activity["count"].quantile(ACTIVITY_QUANTILE))
)

customer_threshold = int(
    round(customer_activity["count"].quantile(ACTIVITY_QUANTILE))
)

active_movies = movie_activity[
    movie_activity["count"] >= movie_threshold
].index

active_customers = customer_activity[
    customer_activity["count"] >= customer_threshold
].index

filtered_ratings = ratings[
    ratings["Movie_Id"].isin(active_movies)
    & ratings["Cust_Id"].isin(active_customers)
].copy()

print("Movie threshold:", movie_threshold)
print("Customer threshold:", customer_threshold)
print("Original rows:", f"{len(ratings):,}")
print("Filtered rows:", f"{len(filtered_ratings):,}")


## 8. Load Movie Titles


In [ ]:
titles = pd.read_csv(
    TITLES_PATH,
    encoding="ISO-8859-1",
    header=None,
    names=["Movie_Id", "Year", "Name"],
    on_bad_lines="skip"
)

titles["Movie_Id"] = pd.to_numeric(
    titles["Movie_Id"],
    errors="coerce"
)

titles = titles.dropna(subset=["Movie_Id"]).copy()
titles["Movie_Id"] = titles["Movie_Id"].astype(int)

titles.head()


## 9. Prepare Data for SVD

`scikit-surprise` expects user, item and rating columns.

For quick portfolio execution, cross-validation can be performed on a reproducible sample.
The final model can then be trained on the complete filtered dataset.


In [ ]:
EVAL_ROWS = min(100_000, len(filtered_ratings))

eval_ratings = filtered_ratings.sample(
    EVAL_ROWS,
    random_state=42
)

reader = Reader(rating_scale=(1, 5))

surprise_data = Dataset.load_from_df(
    eval_ratings[["Cust_Id", "Movie_Id", "Rating"]],
    reader
)


## 10. Cross-Validate the SVD Model

RMSE and MAE measure rating-prediction error. Lower values indicate smaller prediction errors.


In [ ]:
svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

cv_results = cross_validate(
    svd,
    surprise_data,
    measures=["RMSE", "MAE"],
    cv=3,
    verbose=False
)

cv_summary = pd.DataFrame({
    "Fold": range(1, len(cv_results["test_rmse"]) + 1),
    "RMSE": cv_results["test_rmse"],
    "MAE": cv_results["test_mae"]
})

cv_summary


In [ ]:
cv_summary[["RMSE", "MAE"]].mean().round(4)


## 11. Train the Final SVD Model


In [ ]:
full_data = Dataset.load_from_df(
    filtered_ratings[
        ["Cust_Id", "Movie_Id", "Rating"]
    ],
    reader
)

trainset = full_data.build_full_trainset()

final_svd = SVD(
    n_factors=100,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

final_svd.fit(trainset)


## 12. Generate Personalized Recommendations

Instead of hard-coding one customer throughout the notebook, the function below can generate
recommendations for any customer present in the filtered ratings data.


In [ ]:
def recommend_movies(
    customer_id,
    model,
    ratings_df,
    titles_df,
    top_n=10
):
    rated_movies = set(
        ratings_df.loc[
            ratings_df["Cust_Id"] == customer_id,
            "Movie_Id"
        ]
    )

    candidates = titles_df[
        ~titles_df["Movie_Id"].isin(rated_movies)
    ].copy()

    candidates["Estimated_Rating"] = (
        candidates["Movie_Id"]
        .apply(lambda movie_id: model.predict(customer_id, movie_id).est)
    )

    return (
        candidates[
            ["Movie_Id", "Name", "Year", "Estimated_Rating"]
        ]
        .sort_values("Estimated_Rating", ascending=False)
        .head(top_n)
        .reset_index(drop=True)
    )


In [ ]:
sample_customer = int(
    filtered_ratings["Cust_Id"]
    .value_counts()
    .index[0]
)

recommendations = recommend_movies(
    customer_id=sample_customer,
    model=final_svd,
    ratings_df=filtered_ratings,
    titles_df=titles,
    top_n=10
)

print("Customer:", sample_customer)
recommendations


## 13. Conclusion

This project demonstrates a complete collaborative-filtering workflow:

- parsing a non-standard ratings format
- understanding user/item sparsity
- filtering low-activity entities
- evaluating an SVD recommender with cross-validation
- training a final model
- ranking unseen movies for a customer

### Limitations

- recommendations use explicit ratings only
- SVD does not use movie metadata such as genre, cast or text
- activity filtering changes the population used for modeling
- recommendation quality should ideally also be evaluated with ranking-oriented metrics
  such as Precision@K or Recall@K

A natural next step would be a hybrid recommender that combines collaborative filtering
with content features.
